In [1]:
# === Addestramento esperti: UNICO SCALER + CLASSI GLOBALI ===
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, Concatenate
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf, random as rn, joblib

os.makedirs("expert_models", exist_ok=True)

file_list = [
    "PSD_dataset_Bergantin_Fulvio_42.csv",
    "PSD_dataset_Cambareri_Chiara_26.csv",
    "PSD_dataset_Capria_Francesco_26.csv",
    "PSD_dataset_Capria_Alessandro_19.csv",
    "PSD_dataset_Cicirelli_Franco_50.csv",
    "PSD_dataset_Coppolillo_Erica_25.csv",
    "PSD_dataset_Damore_Francesco_45.csv",
    "PSD_dataset_DeMaio_Annarita_34.csv",
    "PSD_dataset_Falcone_Alberto_40.csv",
    "PSD_dataset_Folino_Gianluigi_52.csv",
    "PSD_dataset_Guarascio_Massimo_42.csv",
    "PSD_dataset_Guerrieri_Antonio_42.csv",
    "PSD_dataset_Islam_Md_Babul.csv",
    "PSD_dataset_Khan_Irfanullah_33.csv",
    "PSD_dataset_Macrì_Davide_44.csv"
]

# Seed
seed=230782
os.environ['PYTHONHASHSEED']='0'; np.random.seed(seed); tf.random.set_seed(seed); rn.seed(seed)
tf.config.threading.set_inter_op_parallelism_threads(1); tf.config.threading.set_intra_op_parallelism_threads(1)
os.environ['TF_DETERMINISTIC_OPS']='1'; os.environ['TF_CUDNN_DETERMINISTIC']='1'

def create_resnet(input_dim, num_classes, dropout_pcg=0.25, factors=256):
    inp = Input(shape=(input_dim,))
    out = Dense(factors, activation="relu")(inp)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)
    res = Dense(factors, activation="relu")(out)
    res = BatchNormalization()(res)
    res = Dropout(dropout_pcg)(res)
    out = Add()([out, res])
    out = Concatenate()([out, inp])
    out = Dense(num_classes, activation="softmax")(out)
    m = Model(inp, out)
    m.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.RMSprop(1e-3), metrics=['accuracy'])
    return m

# ---------- PASS 1: classi globali + scaler globale sui TRAIN ----------
all_labels=set()
train_chunks=[]; input_dim=None
for file in file_list:
    df=pd.read_csv(file)
    X=df.drop("Event Id",axis=1).astype(float).values
    y=df["Event Id"].values
    if input_dim is None: input_dim=X.shape[1]
    Xtr, Xte, ytr, yte = train_test_split(X,y,train_size=0.8,shuffle=True,stratify=y,random_state=42)
    train_chunks.append(Xtr)
    all_labels.update(np.unique(y))

classes_global = np.array(sorted(all_labels))
NUM_CLASSES = len(classes_global)
scaler = MinMaxScaler().fit(np.vstack(train_chunks))  # unico scaler sui TRAIN di tutti

joblib.dump(scaler, "expert_models/global_minmax_scaler.joblib")
np.save("expert_models/classes.npy", classes_global)

# ---------- PASS 2: training esperti con scaler & classi globali ----------
for file in file_list:
    name = os.path.splitext(os.path.basename(file))[0]
    print(f"\n=== Training modello per: {name} ===")
    psd_df = pd.read_csv(file)
    X = psd_df.drop("Event Id", axis=1).astype(float).values
    y = psd_df["Event Id"].values

    Xtr, Xte, ytr, yte = train_test_split(X,y,train_size=0.8,shuffle=True,stratify=y,random_state=42)

    # Trasforma con SCALER GLOBALE
    Xtr_s = scaler.transform(Xtr)
    Xte_s = scaler.transform(Xte)

    # Oversampling sul train
    df_tr = pd.DataFrame(Xtr_s); df_tr["Event Id"]=ytr
    max_n = df_tr["Event Id"].value_counts().max()
    dfs=[]
    for cls, sub in df_tr.groupby("Event Id"):
        if len(sub) < max_n:
            sub = resample(sub, replace=True, n_samples=max_n, random_state=42)
        dfs.append(sub)
    df_bal = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    Xtr_bal = df_bal.drop("Event Id",axis=1).values
    ytr_bal = df_bal["Event Id"].values

    # One-hot con CLASSI GLOBALI (ordine identico per tutti)
    to_index = {c:i for i,c in enumerate(classes_global)}
    ytr_cat = to_categorical([to_index[c] for c in ytr_bal], num_classes=NUM_CLASSES)
    yte_idx = np.array([to_index[c] for c in yte])  # per valutazione

    model = create_resnet(input_dim, NUM_CLASSES)
    ckpt = ModelCheckpoint(f"expert_models/best_{name}.weights.h5",
                           monitor="val_loss", save_best_only=True, save_weights_only=True, mode="min", verbose=0)
    rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)

    model.fit(Xtr_bal, ytr_cat, validation_split=0.2, epochs=150, batch_size=64,
              callbacks=[ckpt, rlr], verbose=0)

    model.load_weights(f"expert_models/best_{name}.weights.h5")
    ypred = model.predict(Xte_s, verbose=0)
    ypred_lab = np.argmax(ypred, axis=1)
    acc = accuracy_score(yte_idx, ypred_lab)
    print(f"Accuracy: {acc:.2f}")
    print(classification_report(yte_idx, ypred_lab, target_names=[str(c) for c in classes_global]))



=== Training modello per: PSD_dataset_Bergantin_Fulvio_42 ===
Accuracy: 0.94
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93       128
     33024.0       0.98      0.93      0.95        45
     33025.0       0.93      0.96      0.95        45
     33026.0       0.98      0.93      0.95        45
     33027.0       0.91      0.91      0.91        45

    accuracy                           0.94       308
   macro avg       0.94      0.93      0.94       308
weighted avg       0.94      0.94      0.94       308


=== Training modello per: PSD_dataset_Cambareri_Chiara_26 ===
Accuracy: 0.92
              precision    recall  f1-score   support

         0.0       0.90      0.94      0.92       128
     33024.0       0.98      0.89      0.93        45
     33025.0       0.90      0.96      0.92        45
     33026.0       0.90      0.84      0.87        45
     33027.0       0.95      0.91      0.93        45

    accuracy                 